In [1]:
# =========================
# Recommender-Systems-LLM
# Full fresh Colab smoke test from GitHub
# =========================

import os
import time
import json
import shutil
import subprocess
import urllib.request
from pathlib import Path

# -------------------------
# 1) Clean old copy if present
# -------------------------
REPO_DIR = Path("/content/Recommender-Systems-LLM")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# -------------------------
# 2) Clone your GitHub repo
# -------------------------
!git clone https://github.com/AngshulMajumdar/Recommender-Systems-LLM.git /content/Recommender-Systems-LLM

assert REPO_DIR.exists(), f"Repo was not cloned: {REPO_DIR}"
assert (REPO_DIR / "app").exists(), "Missing app/ folder"
assert (REPO_DIR / "requirements.txt").exists(), "Missing requirements.txt"

print("Repo cloned to:", REPO_DIR)
print("\nTop-level contents:")
for p in sorted(REPO_DIR.iterdir()):
    print("-", p.name)

# -------------------------
# 3) Install requirements
# -------------------------
os.chdir(REPO_DIR)
!pip -q install -r requirements.txt

# -------------------------
# 4) Start FastAPI server
# -------------------------
SERVER_LOG = "/content/ml100k_api_server.log"
if os.path.exists(SERVER_LOG):
    os.remove(SERVER_LOG)

proc = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(REPO_DIR),
    stdout=open(SERVER_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print("\nStarted uvicorn PID:", proc.pid)

# -------------------------
# 5) Wait for health endpoint
# -------------------------
import requests

health_url = "http://127.0.0.1:8000/health"
ok = False

for _ in range(60):
    try:
        r = requests.get(health_url, timeout=5)
        if r.status_code == 200:
            print("\nHealth check passed:")
            print(r.text)
            ok = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ok:
    print("\n===== FULL SERVER LOG =====")
    with open(SERVER_LOG, "r") as f:
        print(f.read())
    raise RuntimeError("Server did not start.")

print("\nSwagger docs:")
print("http://127.0.0.1:8000/docs")

# -------------------------
# 6) Download official MovieLens 100K zip
# -------------------------
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATASET_ZIP_PATH = Path("/content/ml-100k.zip")

if not DATASET_ZIP_PATH.exists():
    print("\nDownloading MovieLens 100K...")
    urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP_PATH)
    print("Downloaded to:", DATASET_ZIP_PATH)
else:
    print("\nDataset already exists:", DATASET_ZIP_PATH)

assert DATASET_ZIP_PATH.exists(), "Dataset download failed."

# -------------------------
# 7) 20-row smoke test with backend=mock
# -------------------------
run_url = "http://127.0.0.1:8000/run"

with open(DATASET_ZIP_PATH, "rb") as f:
    files = {"dataset": (DATASET_ZIP_PATH.name, f, "application/zip")}
    data = {
        "backend": "mock",
        "positive_threshold": "4",
        "max_history": "10",
        "max_rows": "20",
        "top_k_json": "[5,10,20]",
        "use_4bit_if_available": "true",
        "save_outputs": "true",
    }
    resp = requests.post(run_url, files=files, data=data, timeout=600)

print("\nSmoke test status:", resp.status_code)

try:
    out = resp.json()
    print(json.dumps(out, indent=2)[:6000])
except Exception:
    print(resp.text[:6000])
    raise

if resp.status_code != 200:
    print("\n===== SERVER LOG TAIL =====")
    !tail -n 100 /content/ml100k_api_server.log
    raise RuntimeError("20-row smoke test failed.")

# -------------------------
# 8) Show runs
# -------------------------
runs_resp = requests.get("http://127.0.0.1:8000/runs", timeout=30)
print("\nRuns endpoint:")
print(runs_resp.text[:4000])

print("\nDone.")
print("Docs: http://127.0.0.1:8000/docs")

Cloning into '/content/Recommender-Systems-LLM'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 60 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 43.49 KiB | 1.45 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Repo cloned to: /content/Recommender-Systems-LLM

Top-level contents:
- .git
- LICENSE
- README.md
- app
- examples
- requirements.txt
- sample_data
- tests
- tmp_runs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.2 MB/s eta 0:00:00

Started uvicorn PID: 644

Health check passed:
{"status":"ok"}

Swagger docs:
http://127.0.0.1:8000/docs

Downloaded to: /content/ml-100k.zip

Smoke test status: 200
{
  "status": "ok",
  "message": "Pipeline completed successfully.",
  "run_dir": "/content/Recommender-Systems-LLM/runs/run_20260323_151316",
  "config": {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "positive_threshold": 4